# Phase 3: Linking every neighbourhood to its nearest station

In this notebook we find the nearest railway station for every neighbourhood. We then add the station code, name and type to the neighbourhood table. The station code is used later to add the disruption numbers (Phase 4 and 5).

In [1]:
import os                  # for file paths
import pandas as pd        # for tables
import geopandas as gpd    # for maps and distances between places

# Folder with the clean data from Phase 1 and 2
clean_folder = 'data/cleaned_data'

# Load the clean neighbourhood table, the station list and the neighbourhood map
buurten = pd.read_pickle(os.path.join(clean_folder, 'buurten_schoon.pkl'))
stations = pd.read_pickle(os.path.join(clean_folder, 'stations.pkl'))
buurten_kaart = gpd.read_file(os.path.join(clean_folder, 'buurten_kaart.gpkg'))

print('buurten:', len(buurten), 'rows')
print('stations:', len(stations), 'rows')
print('buurten_kaart:', len(buurten_kaart), 'rows')

c:\Users\alecm\anaconda3\envs\TIL6022-26\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


buurten: 11221 rows
stations: 397 rows
buurten_kaart: 14515 rows


## Step 1: One point per neighbourhood
To measure which station is closest, we need one point per neighbourhood. We use the centre of each neighbourhood shape (the *centroid*).

In [2]:
# The map has more neighbourhoods than our clean table.
# we linked the map to the clean table on buurtcode, so that only the neighbourhoods in our clean data where kept.
buurt_punten = buurten_kaart.merge(buurten[['buurtcode']], on='buurtcode')

# Replace the shape of every neighbourhood by its centre point.
# So instead of an area, every neighbourhood is now one dot on the map.
buurt_punten['geometry'] = buurt_punten.centroid

print('Neighbourhood points:', len(buurt_punten))
buurt_punten.head()

Neighbourhood points: 11221


,buurtcode,buurtnaam,gemeentenaam,geometry
0,BU00140000,Binnenstad-Noord,Groningen,POINT (233655.736 582070.787)
1,BU00140001,Binnenstad-Zuid,Groningen,POINT (233834.594 581588.756)
2,BU00140002,Binnenstad-Oost,Groningen,POINT (234292.167 582097.291)
3,BU00140003,Binnenstad-West,Groningen,POINT (233195.739 581682.521)
4,BU00140005,Hortusbuurt-Ebbingekwartier,Groningen,POINT (233458.532 582523.152)


## Step 2: One point per station
The station list only has two number columns with the location (latitude and longitude, in degrees). We turn these into points on a map, in the same coordinate system as the neighbourhood map.

In [3]:
# Give the station columns clearer names (axis=1 means: rename columns)
stations = stations.rename({'code': 'station_code', 'name_long': 'station_naam', 'type': 'station_type'}, axis=1)

# Make a point from the longitude (x, east-west) and latitude (y, north-south) of every station
punten = gpd.points_from_xy(stations['geo_lng'], stations['geo_lat'])

# Put the station table and the points together in a map table (GeoDataFrame).
# crs='EPSG:4326' tells GeoPandas that these coordinates are in degrees.
station_punten = gpd.GeoDataFrame(stations, geometry=punten, crs='EPSG:4326')

# Change the coordinates from degrees to metres (EPSG:28992), the same system as the neighbourhood map.
station_punten = station_punten.to_crs('EPSG:28992')

# Keep only the columns we need
station_punten = station_punten[['station_code', 'station_naam', 'station_type', 'geometry']]

station_punten.head()

,station_code,station_naam,station_type,geometry
0,HT,'s-Hertogenbosch,knooppuntIntercitystation,POINT (148529.229 411304.081)
1,HTO,'s-Hertogenbosch Oost,stoptreinstation,POINT (150239.092 412422.932)
2,HDE,'t Harde,stoptreinstation,POINT (189457.906 491380.869)
3,ATN,Aalten,stoptreinstation,POINT (236958.504 437654.779)
4,AC,Abcoude,stoptreinstation,POINT (127005.741 476800.771)


## Step 3: Find the nearest station for every neighbourhood

In [4]:
# For every neighbourhood point, GeoPandas looks for the station point that is closest by
gekoppeld = gpd.sjoin_nearest(buurt_punten, station_punten)

# Check how many neighbourhoods were linked to a station
print('Linked neighbourhoods:', len(gekoppeld))
gekoppeld.head()

Linked neighbourhoods: 11221


,buurtcode,buurtnaam,gemeentenaam,geometry,index_right,station_code,station_naam,station_type
0,BU00140000,Binnenstad-Noord,Groningen,POINT (233655.736 582070.787),154,GN,Groningen,knooppuntIntercitystation
1,BU00140001,Binnenstad-Zuid,Groningen,POINT (233834.594 581588.756),154,GN,Groningen,knooppuntIntercitystation
2,BU00140002,Binnenstad-Oost,Groningen,POINT (234292.167 582097.291),154,GN,Groningen,knooppuntIntercitystation
3,BU00140003,Binnenstad-West,Groningen,POINT (233195.739 581682.521),154,GN,Groningen,knooppuntIntercitystation
4,BU00140005,Hortusbuurt-Ebbingekwartier,Groningen,POINT (233458.532 582523.152),156,GNN,Groningen Noord,stoptreinstation


## Step 4: Major or minor station
For SQ3 we compare major stations with minor stations. We count megastations, intercity hub stations and intercity stations as **major**; all other stations (local stations) are **minor**.

In [5]:
# True if the station is one of the three major types, False if it is a minor (local) station.
gekoppeld['station_groot'] = (
    (gekoppeld['station_type'] == 'megastation')
    | (gekoppeld['station_type'] == 'knooppuntIntercitystation')
    | (gekoppeld['station_type'] == 'intercitystation')
)

# Count how many neighbourhoods have a major (True) or minor (False) station as nearest station
gekoppeld['station_groot'].value_counts()

station_groot
False    8677
True     2544
Name: count, dtype: int64

## Step 5: Add the station information to the neighbourhood table

In [6]:
# Take only the columns we need from the linked table
station_info = gekoppeld[['buurtcode', 'station_code', 'station_naam', 'station_type', 'station_groot']]

# Add the station information to the clean neighbourhood table, matched on buurtcode
buurten_met_station = buurten.merge(station_info, on='buurtcode')

# Check: the number of neighbourhoods should be the same as at the start,
# and no neighbourhood should be without a station (isna() is True for an empty cell, sum() counts the Trues)
print('Neighbourhoods:', len(buurten_met_station))
print('Neighbourhoods without a station:', buurten_met_station['station_code'].isna().sum())

# Save the table for the next phase
buurten_met_station.to_pickle(os.path.join(clean_folder, 'buurten_met_station.pkl'))

Neighbourhoods: 11221
Neighbourhoods without a station: 0
